# Linear regression — pure NumPy

**Dataset:** `diabetes.csv` (canonical Diabetes progression, 442 rows × 10 features). Ye file app ke offline kernel me already seeded hai.

**Rule:** split pehle, scaler sirf training rows par fit, aur test ko sirf final score ke liye chhoona. Har number ka shape ya unit batao.

## 1. Load the canonical CSV

In [3]:
import numpy as np

raw = np.genfromtxt('diabetes.csv', delimiter=',', names=True)
feature_names = list(raw.dtype.names[:-1])
X = np.column_stack([raw[name] for name in feature_names]).astype(float)
y = np.asarray(raw['target'], dtype=float)

print('rows:', X.shape[0], '| features:', X.shape[1])
print('target mean:', round(float(y.mean()), 2), '| target std:', round(float(y.std()), 2))
print('first feature:', feature_names[0], '| first value:', X[0, 0])

rows: 442 | features: 10
target mean: 152.13 | target std: 77.01
first feature: age | first value: 59.0

## 2. Split before fitting anything

In [5]:
rng = np.random.default_rng(7)
order = rng.permutation(len(y))
cut = int(0.8 * len(y))
train_idx, test_idx = order[:cut], order[cut:]

print('train:', len(train_idx), '| test:', len(test_idx))
print('train target mean:', round(float(y[train_idx].mean()), 2))
print('test target mean :', round(float(y[test_idx].mean()), 2))

train: 353 | test: 89
train target mean: 153.88
test target mean : 145.21

## 3. Standardise on train only, then solve least squares

In [7]:
mean = X[train_idx].mean(axis=0)
scale = X[train_idx].std(axis=0)
scale[scale == 0] = 1.0
Xs = (X - mean) / scale

Xtr = np.column_stack([np.ones(len(train_idx)), Xs[train_idx]])
Xte = np.column_stack([np.ones(len(test_idx)), Xs[test_idx]])

theta, *_ = np.linalg.lstsq(Xtr, y[train_idx], rcond=None)
pred = Xte @ theta
print('intercept:', round(float(theta[0]), 2))
print('largest absolute weight:', feature_names[int(np.argmax(np.abs(theta[1:])))] )

intercept: 153.88
largest absolute weight: s1

## 4. Metrics — errors in target units, R² as a relative score

In [9]:
def regression_metrics(y_true, y_pred):
    error = y_true - y_pred
    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(np.mean(error ** 2)))
    ss_res = float(np.sum(error ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot
    return mae, rmse, r2

mae, rmse, r2 = regression_metrics(y[test_idx], pred)
print(f'MAE : {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²  : {r2:.3f}')
print('baseline RMSE (predict train mean):', round(float(np.sqrt(np.mean((y[test_idx] - y[train_idx].mean()) ** 2))), 2))

MAE : 46.95
RMSE: 57.16
R²  : 0.477
baseline RMSE (predict train mean): 79.5

## 5. Ridge — penalise large weights without touching the intercept

In [11]:
alpha = 1.0
penalty = alpha * np.eye(Xtr.shape[1])
penalty[0, 0] = 0.0
theta_ridge = np.linalg.solve(Xtr.T @ Xtr + penalty, Xtr.T @ y[train_idx])
ridge_pred = Xte @ theta_ridge

print('ridge MAE:', round(float(np.mean(np.abs(y[test_idx] - ridge_pred))), 2))
print('weight norm ordinary:', round(float(np.linalg.norm(theta[1:])), 2))
print('weight norm ridge   :', round(float(np.linalg.norm(theta_ridge[1:])), 2))

ridge MAE: 47.04
weight norm ordinary: 64.7
weight norm ridge   : 55.82

## 6. Gradient descent — same loss, iterative solution

In [13]:
weights = np.zeros(Xtr.shape[1])
learning_rate = 0.05
losses = []
for epoch in range(1500):
    residual = Xtr @ weights - y[train_idx]
    loss = float(np.mean(residual ** 2))
    losses.append(loss)
    gradient = (2.0 / len(train_idx)) * (Xtr.T @ residual)
    weights -= learning_rate * gradient

gd_pred = Xte @ weights
print('first loss:', round(losses[0], 2), '| last loss:', round(losses[-1], 2))
print('gradient-descent test RMSE:', round(float(np.sqrt(np.mean((y[test_idx] - gd_pred) ** 2))), 2))
print('difference from lstsq weights:', round(float(np.linalg.norm(weights - theta)), 3))

first loss: 29513.73 | last loss: 2771.86
gradient-descent test RMSE: 57.19
difference from lstsq weights: 13.427

## 7. What this model does not prove

- Linear regression assumes an additive linear relationship; it cannot discover interactions on its own.
- The split is random, not temporal. For a real clinical or business decision, split by time or site.
- MSE is sensitive to outliers. Report MAE and inspect the largest residuals.
- A coefficient is conditional on the other features and the sampled population; it is not a causal effect.

**Exercise:** add one interaction feature, refit on train, and compare test RMSE. If it improves, check that the interaction was available at prediction time.